# Homework 2 – Vector Search

Embedding text with a lightweight ONNX model, building vector search from scratch, and comparing keyword vs vector vs hybrid search.

**Knowledge base:** All markdown lesson files from the `llm-zoomcamp` repo at commit `8c1834d`.
**Embedder:** `Xenova/all-MiniLM-L6-v2` via ONNX runtime (no PyTorch / no CUDA needed).

## Setup

The `download.py` and `embedder.py` helper scripts live in this directory alongside the notebook.
Run `python download.py` once to fetch the ONNX model before executing this notebook.

In [1]:
import numpy as np
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch

from embedder import Embedder

embedder = Embedder()

---
## Q1. Embedding a query

Embed the query and inspect the first element of the 384-dimensional vector.

In [2]:
QUERY = "How does approximate nearest neighbor search work?"

v = embedder.encode(QUERY)

print(f"Vector length : {len(v)}")
print(f"v[0]          : {v[0]:.4f}")

Vector length : 384
v[0]          : -0.0206


> **Answer Q1:** `-0.02`

---
## Load the data

Pull all 72 lesson pages from the course repo, pinned to commit `8c1834d`.

In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"Pages loaded: {len(documents)}")

Pages loaded: 72


---
## Q2. Cosine similarity

Normalized vectors → dot product equals cosine similarity.  
We embed `02-vector-search/lessons/07-sqlitesearch-vector.md` and dot it with the Q1 query vector.

In [4]:
TARGET_FILE = "02-vector-search/lessons/07-sqlitesearch-vector.md"

target_doc = next(d for d in documents if d["filename"] == TARGET_FILE)
v_doc = embedder.encode(target_doc["content"])

similarity = float(np.dot(v, v_doc))
print(f"Cosine similarity: {similarity:.4f}")

Cosine similarity: 0.3611


> **Answer Q2:** `0.37`

---
## Q3. Chunking and search by hand

Split pages into overlapping 2000-char chunks (step=1000), embed them all, and score against the Q1 query with a matrix dot product.

In [5]:
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks: {len(chunks)}")

Total chunks: 295


In [6]:
X = embedder.encode_batch([c["content"] for c in chunks])

scores = X.dot(v)
best_idx = int(np.argmax(scores))

print(f"Best score    : {scores[best_idx]:.4f}")
print(f"Best filename : {chunks[best_idx]['filename']}")

Best score    : 0.6489
Best filename : 02-vector-search/lessons/07-sqlitesearch-vector.md


> **Answer Q3:** `02-vector-search/lessons/07-sqlitesearch-vector.md`

---
## Q4. Vector search with minsearch

Index all chunks with `VectorSearch` and run a new query.

In [7]:
chunk_embeddings = embedder.encode_batch([c["content"] for c in chunks])

vs = VectorSearch(keyword_fields=["filename"])
vs.fit(chunk_embeddings, chunks)

In [8]:
Q4_QUERY = "What metric do we use to evaluate a search engine?"

q4_v = embedder.encode(Q4_QUERY)
q4_results = vs.search(q4_v, num_results=5)

print(f"First result: {q4_results[0]['filename']}")
print("\nTop 5:")
for i, r in enumerate(q4_results, 1):
    print(f"  {i}. {r['filename']}")

First result: 04-evaluation/lessons/05-search-metrics.md

Top 5:
  1. 04-evaluation/lessons/05-search-metrics.md
  2. 04-evaluation/lessons/01-intro.md
  3. 01-agentic-rag/lessons/05-search.md
  4. 04-evaluation/lessons/01-intro.md
  5. 04-evaluation/lessons/15-next-steps.md


> **Answer Q4:** `04-evaluation/lessons/05-search-metrics.md`

---
## Q5. Text search vs vector search

Index the same chunks with keyword search and compare top-5 results for a PostgreSQL query.  
We find the file that appears in vector results but not in text results.

In [9]:
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

In [10]:
Q5_QUERY = "How do I store vectors in PostgreSQL?"

q5_v = embedder.encode(Q5_QUERY)
q5_vector_results = vs.search(q5_v, num_results=5)
q5_text_results = text_index.search(Q5_QUERY, num_results=5)

vec_files = {r["filename"] for r in q5_vector_results}
txt_files = {r["filename"] for r in q5_text_results}

print("Vector top-5 (unique filenames):", sorted(vec_files))
print("Text   top-5 (unique filenames):", sorted(txt_files))
print("\nIn vector but NOT in text:", vec_files - txt_files)

Vector top-5 (unique filenames): ['02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md']
Text   top-5 (unique filenames): ['02-vector-search/lessons/01-intro.md', '02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md']

In vector but NOT in text: {'02-vector-search/lessons/08-pgvector.md'}


> **Answer Q5:** `02-vector-search/lessons/08-pgvector.md`

---
## Q6. Hybrid search with Reciprocal Rank Fusion (RRF)

Merge vector and text results using RRF (`k=60`).  
A document found by both searches accumulates contributions from each ranked list.

In [11]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [12]:
Q6_QUERY = "How do I give the model access to tools?"

q6_v = embedder.encode(Q6_QUERY)
q6_vector_results = vs.search(q6_v, num_results=5)
q6_text_results = text_index.search(Q6_QUERY, num_results=5)
q6_hybrid = rrf([q6_vector_results, q6_text_results])

print("Vector top-5:", [r["filename"] for r in q6_vector_results])
print("Text   top-5:", [r["filename"] for r in q6_text_results])
print("\nHybrid top-5 (RRF):")
for i, r in enumerate(q6_hybrid, 1):
    print(f"  {i}. {r['filename']}")

Vector top-5: ['01-agentic-rag/lessons/01-intro.md', '04-evaluation/lessons/02-ground-truth.md', '01-agentic-rag/lessons/16-other-frameworks.md', '01-agentic-rag/lessons/15-frameworks.md', '01-agentic-rag/lessons/13-function-calling.md']
Text   top-5: ['01-agentic-rag/lessons/14-agentic-loop.md', '01-agentic-rag/lessons/13-function-calling.md', '01-agentic-rag/lessons/13-function-calling.md', '01-agentic-rag/lessons/13-function-calling.md', '04-evaluation/lessons/02-ground-truth.md']

Hybrid top-5 (RRF):
  1. 01-agentic-rag/lessons/13-function-calling.md
  2. 01-agentic-rag/lessons/01-intro.md
  3. 01-agentic-rag/lessons/14-agentic-loop.md
  4. 04-evaluation/lessons/02-ground-truth.md
  5. 01-agentic-rag/lessons/16-other-frameworks.md


> **Answer Q6:** `01-agentic-rag/lessons/13-function-calling.md`
>
> This file isn't first in either individual search. It wins because it ranks high in **both** lists.